In [1]:
import subprocess, sys

# ── nvidia-smi ────────────────────────────────────────────────────────────────
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
print("=== GPU ===")
print(result.stdout.strip() if result.returncode == 0 else "nvidia-smi not found")

# ── PyTorch ───────────────────────────────────────────────────────────────────
print("\n=== PyTorch ===")
try:
    import torch
    print(f"Version: {torch.__version__}")
    print(f"CUDA:    {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU:     {torch.cuda.get_device_name(0)}")
        x = torch.rand(1000, 1000, device="cuda")
        torch.matmul(x, x)
        print("Test:    PASSED ✓")
except ImportError:
    print("Not installed")

# ── TensorFlow ────────────────────────────────────────────────────────────────
print("\n=== TensorFlow ===")
try:
    import tensorflow as tf
    print(f"Version: {tf.__version__}")
    gpus = tf.config.list_physical_devices("GPU")
    print(f"GPUs:    {len(gpus)}")
    if gpus:
        with tf.device("/GPU:0"):
            x = tf.random.uniform((1000, 1000))
            tf.matmul(x, x)
        print("Test:    PASSED ✓")
except ImportError:
    print("Not installed")

=== GPU ===
NVIDIA GeForce RTX 5070 Ti, 595.95, 16303 MiB, 12.0

=== PyTorch ===
Version: 2.12.0.dev20260219+cu128
CUDA:    True
GPU:     NVIDIA GeForce RTX 5070 Ti
Test:    PASSED ✓

=== TensorFlow ===


2026-06-29 10:36:33.472425: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:36:33.763870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/harris/miniconda3/envs/ai/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to con

Version: 2.20.0
GPUs:    1
Test:    PASSED ✓


W0000 00:00:1782747396.434059    3061 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1782747396.435733    3061 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1782747396.441369    3061 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13177 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0


In [2]:
import torch
import time

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda")

# large matrices to stress GPU
size = 10000

print("\nAllocating tensors on GPU...")
a = torch.rand(size, size, device=device)
b = torch.rand(size, size, device=device)

# warmup (important for GPU timing)
torch.matmul(a, b)
torch.cuda.synchronize()

print("Running benchmark...")

start = time.time()

c = torch.matmul(a, b)

torch.cuda.synchronize()
end = time.time()

print(f"Matrix size: {size} x {size}")
print(f"Execution time: {end - start:.4f} seconds")

print("\nGPU memory usage:")
print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved :", torch.cuda.memory_reserved() / 1024**3, "GB")

PyTorch version: 2.12.0.dev20260219+cu128
CUDA available: True

Allocating tensors on GPU...
Running benchmark...
Matrix size: 10000 x 10000
Execution time: 0.0627 seconds

GPU memory usage:
Allocated: 1.1270751953125 GB
Reserved : 1.138671875 GB


In [3]:
# PyTorch GPU Check
import torch

print("PyTorch GPU Check")
print("-" * 40)
print(f"Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Test GPU calculation
    x = torch.rand(1000, 1000, device='cuda')
    y = x @ x
    print(f"✓ GPU calculation successful on {x.device}")
else:
    print("✗ No CUDA GPU detected")

PyTorch GPU Check
----------------------------------------
Version: 2.12.0.dev20260219+cu128
CUDA available: True
GPU name: NVIDIA GeForce RTX 5070 Ti
GPU memory: 17.1 GB
✓ GPU calculation successful on cuda:0


In [4]:
import subprocess, sys

MIN_DRIVER   = 570.0
MIN_CUDA     = (12, 8)
MIN_TORCH    = (2, 6)
BLACKWELL_CC = (10, 0)

def parse_version(v: str):
    """Return first two numeric components as a tuple of ints."""
    parts = []
    for p in v.strip().split("."):
        digits = "".join(c for c in p if c.isdigit())
        if digits:
            parts.append(int(digits))
        if len(parts) == 2:
            break
    return tuple(parts)

# ── nvidia-smi ────────────────────────────────────────────────────────────────
print("=== GPU (nvidia-smi) ===")
result = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,driver_version,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True
)

gpu_name = ""
compute_cap = (0, 0)

if result.returncode != 0:
    print("nvidia-smi not found — cannot detect GPU")
else:
    for line in result.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 4:
            continue
        gpu_name, driver_ver, vram, cc_str = parts[0], parts[1], parts[2], parts[3]
        compute_cap = parse_version(cc_str)

        print(f"  GPU:              {gpu_name}")
        print(f"  Driver version:   {driver_ver}")
        print(f"  VRAM:             {vram}")
        print(f"  Compute cap:      {cc_str}")

        # ── Blackwell detection ───────────────────────────────────────────────
        if compute_cap >= BLACKWELL_CC:
            print("  Architecture:     ✓ Blackwell (sm_100+) detected")
        elif compute_cap >= (9, 0):
            print("  Architecture:     Ada Lovelace / Hopper (sm_90) — not Blackwell")
        elif compute_cap >= (8, 9):
            print("  Architecture:     Ada Lovelace (sm_89)")
        elif compute_cap >= (8, 6):
            print("  Architecture:     Ampere (sm_86) — your RTX 3060")
        else:
            print(f"  Architecture:     Older architecture (sm_{cc_str})")

        # ── Driver version check ──────────────────────────────────────────────
        try:
            drv_float = float(driver_ver.split(".")[0] + "." +
                              driver_ver.split(".")[1] if "." in driver_ver else driver_ver)
        except ValueError:
            drv_float = 0.0

        if compute_cap >= BLACKWELL_CC and drv_float < MIN_DRIVER:
            print(f"  ⚠ Driver {driver_ver} is below minimum {MIN_DRIVER} for Blackwell")
        elif compute_cap >= BLACKWELL_CC:
            print(f"  ✓ Driver {driver_ver} meets Blackwell requirement (≥{MIN_DRIVER})")

# ── PyTorch ───────────────────────────────────────────────────────────────────
print("\n=== PyTorch ===")
try:
    import torch

    torch_ver = parse_version(torch.__version__)
    cuda_available = torch.cuda.is_available()
    print(f"  Version:          {torch.__version__}")
    print(f"  CUDA available:   {cuda_available}")

    # CUDA runtime version
    cuda_ver = parse_version(torch.version.cuda or "0.0")
    print(f"  CUDA runtime:     {torch.version.cuda}")

    if compute_cap >= BLACKWELL_CC:
        if torch_ver < MIN_TORCH:
            print(f"  ⚠ PyTorch {torch.__version__} may lack Blackwell kernels — upgrade to 2.6+")
        else:
            print(f"  ✓ PyTorch version is Blackwell-compatible")

        if cuda_ver < MIN_CUDA:
            print(f"  ⚠ CUDA {torch.version.cuda} is below 12.8 — required for Blackwell")
        else:
            print(f"  ✓ CUDA runtime meets Blackwell requirement (≥12.8)")

    if cuda_available:
        dev = torch.cuda.get_device_name(0)
        print(f"  Device:           {dev}")

        # Larger stress test — catches memory/driver issues better
        print("  Running matmul stress test (4096×4096 fp16)...")
        x = torch.rand(4096, 4096, dtype=torch.float16, device="cuda")
        y = torch.rand(4096, 4096, dtype=torch.float16, device="cuda")
        torch.cuda.synchronize()
        z = torch.matmul(x, y)
        torch.cuda.synchronize()
        print(f"  Matmul test:      PASSED ✓  (output shape {tuple(z.shape)})")

        # torch.compile check (Blackwell depends on Triton for peak perf)
        print("  Testing torch.compile (reduce-overhead)...")
        try:
            @torch.compile(mode="reduce-overhead")
            def compiled_matmul(a, b):
                return torch.matmul(a, b)
            _ = compiled_matmul(x, y)
            torch.cuda.synchronize()
            print("  torch.compile:    PASSED ✓")
        except Exception as e:
            print(f"  torch.compile:    FAILED ✗ — {e}")

        # Check if GPU is actually running on CUDA vs CPU fallback
        print(f"  Tensor device:    {z.device}")

except ImportError:
    print("  Not installed")

# ── TensorFlow ────────────────────────────────────────────────────────────────
print("\n=== TensorFlow ===")
try:
    import tensorflow as tf

    print(f"  Version:          {tf.__version__}")
    gpus = tf.config.list_physical_devices("GPU")
    print(f"  GPUs detected:    {len(gpus)}")

    if gpus:
        # Check TF isn't silently falling back to CPU
        print("  Running matmul stress test (4096×4096 float32)...")
        with tf.device("/GPU:0"):
            x = tf.random.uniform((4096, 4096), dtype=tf.float32)
            result_tf = tf.matmul(x, x)
        print(f"  Matmul test:      PASSED ✓  (output shape {tuple(result_tf.shape)})")

        if compute_cap >= BLACKWELL_CC:
            print("  ⚠ TensorFlow Blackwell support may be limited — verify your build")
    else:
        print("  ⚠ No GPU detected by TensorFlow — may be running on CPU silently")
        if compute_cap >= BLACKWELL_CC:
            print("  ⚠ Blackwell requires a TF build compiled with CUDA 12.8+")

except ImportError:
    print("  Not installed")

=== GPU (nvidia-smi) ===
  GPU:              NVIDIA GeForce RTX 5070 Ti
  Driver version:   595.95
  VRAM:             16303 MiB
  Compute cap:      12.0
  Architecture:     ✓ Blackwell (sm_100+) detected
  ✓ Driver 595.95 meets Blackwell requirement (≥570.0)

=== PyTorch ===
  Version:          2.12.0.dev20260219+cu128
  CUDA available:   True
  CUDA runtime:     12.8
  ✓ PyTorch version is Blackwell-compatible
  ✓ CUDA runtime meets Blackwell requirement (≥12.8)
  Device:           NVIDIA GeForce RTX 5070 Ti
  Running matmul stress test (4096×4096 fp16)...
  Matmul test:      PASSED ✓  (output shape (4096, 4096))
  Testing torch.compile (reduce-overhead)...
  torch.compile:    PASSED ✓
  Tensor device:    cuda:0

=== TensorFlow ===
  Version:          2.20.0
  GPUs detected:    1
  Running matmul stress test (4096×4096 float32)...
  Matmul test:      PASSED ✓  (output shape (4096, 4096))
  ⚠ TensorFlow Blackwell support may be limited — verify your build
